# Evaluate the selected joint clustering

This notebook evaluates one clustering produced by `01_full_pipeline.ipynb`. It contains the gene-margin, integration, cohesion, diversity, and diagnostic-plot workflow without any hyperparameter search.


In [ ]:
from pathlib import Path
import sys


def find_pipeline_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "Cross Species Pipeline", cwd.parent, cwd.parent.parent]
    for candidate in candidates:
        if (candidate / "src" / "cross_species").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate Cross Species Pipeline/src/cross_species")


PIPELINE_ROOT = find_pipeline_root()
PROJECT_ROOT = PIPELINE_ROOT.parent
SRC_ROOT = PIPELINE_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
RESULTS_ROOT = PIPELINE_ROOT / "results"
DATA_ROOT = PROJECT_ROOT / "data"
print(f"Pipeline root: {PIPELINE_ROOT}")


In [ ]:
from datetime import datetime
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from cross_species import evaluate_joint_modules


## Load and validate a core run


In [ ]:
from cross_species import load_core_run

CORE_RUN_DIR = None  # Set explicitly, or leave None to use the newest core_* run.
if CORE_RUN_DIR is None:
    candidates = [path for path in (RESULTS_ROOT / "core").glob("core_*") if path.is_dir()]
    if not candidates:
        raise FileNotFoundError("No core run found. Run 01_full_pipeline.ipynb first.")
    CORE_RUN_DIR = max(candidates, key=lambda path: path.stat().st_mtime)
core = load_core_run(CORE_RUN_DIR)
prof = core["prof"]
C = core["C"]
graph = core["graph"]
labels = core["labels"]
display(pd.Series(core["manifest"], name="value"))


## Load smoothed expression for spatial diversity


In [ ]:
SMOOTHED_HUMAN_PATH = DATA_ROOT / "sm_adata/human_GSM9629357_CS23_E2S1_bin50_sm.h5ad"
SMOOTHED_MOUSE_PATH = DATA_ROOT / "sm_adata/mouse_E1S1_165_bin50_sm.h5ad"
sm_adata_human = ad.read_h5ad(SMOOTHED_HUMAN_PATH)
sm_adata_mouse = ad.read_h5ad(SMOOTHED_MOUSE_PATH)


## Evaluate and save


In [ ]:
evaluation = evaluate_joint_modules(
    labels,
    C,
    graph,
    prof,
    expression_by_species={
        "mouse": sm_adata_mouse,
        "human": sm_adata_human,
    },
    make_plots=True,
    show_plots=True,
    plot_min_module_size=10,
)

display(evaluation["summary"].to_frame())
display(evaluation["integration"])
display(evaluation["modules"].sort_values("n_genes", ascending=False).head(20))

output_dir = RESULTS_ROOT / "evaluation" / datetime.now().strftime("evaluation_%Y%m%d_%H%M%S")
output_dir.mkdir(parents=True, exist_ok=False)
evaluation["genes"].to_csv(output_dir / "genes.csv.gz", index=False, compression="gzip")
evaluation["modules"].to_csv(output_dir / "modules.csv", index=False)
evaluation["integration"].to_csv(output_dir / "integration.csv", index=False)
evaluation["summary"].to_frame("value").to_csv(output_dir / "summary.csv")
plots_dir = output_dir / "plots"
plots_dir.mkdir()
for name, figure in evaluation["figures"].items():
    figure.savefig(plots_dir / f"{name}.png", dpi=250, bbox_inches="tight")
print(f"Evaluation written to: {output_dir}")
